In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Data', 'Census')
    path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [ ]:
df_area = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name='MSAcodes')
df_area['State FIPS' ] = df_area['State FIPS' ].astype(str).apply('{:0>2}'.format)
df_area = df_area[df_area['State FIPS'] != '72']
states = list(df_area['State FIPS'].unique())


In [ ]:
start_time = time.time()

year_start = 2009
year_end   = 2022
years_to_import = range(year_start, year_end+1)

g_ = '?get='

# User inputs for user API key, desired variables and years to import
api_key_ = f"&key={api_key}"
variables_ = 'NAME'



print('Importing place IDs for all states...')
print('')

list_df_states = []

for state in states:

    print('')
    print('State: ' + str(state))
    list_df_years = []
    
    for year in tqdm(years_to_import):
        root_ = f'https://api.census.gov/data/{year}/acs/acs5'
    
        # Specify which geography to import
        location_ = '&for=state%20legislative%20district%20(upper%20chamber):*&in=state:' + str(state)
        
        
        ## Concatenate constructed URL
        query = f"{root_}{g_}{variables_}{location_}{api_key_}"
    
        ## Call data using URL
    
        # Use requests package to call out to the API
        response = requests.get(query).text
        response = ast.literal_eval(response)
        
        # convert parsed response text to pandas df
        df_census = pd.DataFrame(response[1:], columns = response[0])
        
        # apply year tag
        df_census['Year'] = year
        df_census['State FIPS Code'] = state
    
        list_df_years.append(df_census)

    df_state = pd.concat(list_df_years)
    df_state = df_state.reset_index(drop=True)
    list_df_states.append(df_state)


print('')
print('Concatenating all states together...')

df_dist = pd.concat(list_df_states)


print("")
print("Finished!!")
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")

In [ ]:
start_time = time.time()

year_start = 2009
year_end   = 2022
years_to_import = range(year_start, year_end+1)

g_ = '?get='

# User inputs for user API key, desired variables and years to import
api_key_ = f"&key={api_key}"
variables_ = 'NAME'



print('Importing place IDs for all states...')
print('')

list_df_states = []

for state in states:

    print('')
    print('State: ' + str(state))
    list_df_years = []
    
    for year in tqdm(years_to_import):
        try:
            root_ = f'https://api.census.gov/data/{year}/acs/acs5'
        
            # Specify which geography to import
            location_ = '&for=state%20legislative%20district%20(lower%20chamber):*&in=state:' + str(state)
            
            
            ## Concatenate constructed URL
            query = f"{root_}{g_}{variables_}{location_}{api_key_}"
        
            ## Call data using URL
        
            # Use requests package to call out to the API
            response = requests.get(query).text
            response = ast.literal_eval(response)
            
            # convert parsed response text to pandas df
            df_census = pd.DataFrame(response[1:], columns = response[0])
            
            # apply year tag
            df_census['Year'] = year
            df_census['State FIPS Code'] = state
        
            list_df_years.append(df_census)
        except Exception as e: print(e)

    try:
        df_state = pd.concat(list_df_years)
        df_state = df_state.reset_index(drop=True)
        list_df_states.append(df_state)
    except Exception as e: print(e)


print('')
print('Concatenating all states together...')

df_dist2 = pd.concat(list_df_states)


print("")
print("Finished!!")
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")

In [ ]:
df_dist = df_dist.sort_values(['Year', 'state', 'state legislative district (upper chamber)'], ascending = [False, True, True])

In [ ]:
df_dist2 = df_dist2.sort_values(['Year', 'state', 'state legislative district (lower chamber)'], ascending = [False, True, True])

In [ ]:
df_dist

In [ ]:
df_dist2

In [ ]:
# with pd.ExcelWriter(os.path.join(path_config0, 'Area Codes.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#             df_dist.to_excel(writer, index = False, sheet_name = 'SLUDcodes')

In [ ]:
# with pd.ExcelWriter(os.path.join(path_config0, 'Area Codes.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#             df_dist2.to_excel(writer, index = False, sheet_name = 'SLLDcodes')